# Does the correct CDR-H3 exist anywhere in SimpleFold's distribution?

Every steering method -- density guidance, activation probes, best-of-N selection -- **re-weights
the prior**. All of them fail together if the prior has no mass near the answer. This measures that
directly, before anything is built.

**Design.** 30 nanobodies SimpleFold got badly wrong (>6 A CDR-H3) with a sound framework (<1.5 A,
which excludes the stretched-reference artifacts), plus 10 it got right (<1.5 A) as controls.
Fold each **N=20** times at **tau in {0.01, 0.8, 1.5, 0.3}** and ask: does **any** sample land
within 2 A of the crystal loop?

**tau > 0.3 has never been tested in this project.** tau is the diversity knob; if the correct mode
is rare rather than absent, high tau is what finds it.

**Read-out.** hit rate at 1.5 / 2.0 / 2.5 / 3.0 A, the best-of-k curve, and chain validity --
because a sample can reach 2 A RMSD and still be a broken molecule.

Arms run in the order `0.01, 0.8, 1.5, 0.3` and each is scored and dumped before the next starts,
so a wall-clock timeout still leaves the anchor and the untested high tau on disk.


In [ ]:
import os, sys, json, time, glob, subprocess, shutil
import numpy as np

HARD = ['21TWA1', '6X05K', '6X07B', '7A4TA', '7A4YC', '7B27DDD', '7D4BB', '7MJHF', '7QHAC', '7S7RB', '7SL9B', '7UNZA', '7XKAB', '7XL1B', '7Z1XF', '8ELOB', '8ELQB', '8F6VA', '8GHRD', '8Q7OA', '8S5WB', '8XKID', '8Y9UB', '9BA0N', '9H1JD', '9HO6C', '9IQPB', '9J3JB', '9MY8A', '9PYMB']
EASY = ['6Z20D', '7MFUE', '7P77A', '7SAIC', '7Z1AE', '8G78H', '8H5UB', '9EA9F', '9HVIH', '9R0BC']
PICKS = sorted(HARD + EASY)
TAUS  = ['0.01', '0.8', '1.5', '0.3']      # anchor first, then the untested high end
NS, STEPS, SEED, MODEL = '20', '500', '12345', 'simplefold_100M'
WORK, SCRATCH = '/kaggle/working', '/tmp/sf'
T0 = time.time()
os.makedirs(SCRATCH, exist_ok=True)

R = {'question': 'does the correct CDR-H3 exist anywhere in the prior?',
     'model': MODEL, 'n_samples': int(NS), 'taus': TAUS, 'steps': int(STEPS), 'seed': int(SEED),
     'hard': HARD, 'easy': EASY, 'arms': {}}

def note(tag):
    R['stage'] = tag
    R['elapsed_min'] = round((time.time() - T0) / 60, 1)
    json.dump(R, open(f'{WORK}/hitrate_results.json', 'w'), indent=1, default=str)

import torch
n = torch.cuda.device_count(); cap = torch.cuda.get_device_capability(0)
R['gpus'] = [torch.cuda.get_device_name(g) for g in range(n)]
assert n == 2, f'need T4 x2, got {n}: {R["gpus"]}'
assert cap >= (7, 0), f'{R["gpus"][0]} is sm_{cap[0]}{cap[1]}; this torch build needs sm_70+'

_ds = glob.glob('/kaggle/input/**/targets_vhh.json', recursive=True)
assert len(_ds) == 1, f'targets_vhh.json: expected 1, found {_ds}'
ALL = {f"{t['pdb']}{t['H']}": t for t in json.load(open(_ds[0]))}
TG = {k: ALL[k] for k in PICKS}
assert len(TG) == len(PICKS), f'missing targets: {set(PICKS) - set(ALL)}'
print(R['gpus'], len(TG), 'targets,', len(TAUS), 'tau arms x', NS, 'samples', flush=True)
note('setup')

def must(c, w):
    p = subprocess.run(c, shell=True, text=True, capture_output=True)
    if p.returncode:
        raise RuntimeError(f'{w} failed (rc={p.returncode}):\n' + (p.stdout + p.stderr)[-2000:])

must('git clone -q --depth 1 -b fix-multichain-fasta-parsing '
     'https://github.com/omar-A-hassan/ml-simplefold ' + SCRATCH + '/repo', 'clone')
must(f'{sys.executable} -m pip install -q -e {SCRATCH}/repo', 'install simplefold')
must(f'{sys.executable} -m pip install -q git+https://github.com/facebookresearch/esm.git', 'install esm')
must(f'{sys.executable} -m pip install -q biotite', 'install biotite')

src = open(f'{SCRATCH}/repo/src/simplefold/utils/fasta_utils.py').read()
assert 'chr(ord("A") + i)' in src, 'multichain fix missing -- wrong branch?'

INF = f'{SCRATCH}/repo/src/simplefold/inference.py'; s = open(INF).read()
a = '        esm_model = esm_model.to(device)'
assert s.count(a) == 1, f'fp16 anchor found {s.count(a)}x -- upstream moved it'
open(INF, 'w').write(s.replace(a, '        esm_model = esm_model.half().to(device)'))
print('fp16 patched', flush=True)
note('installed')


## Fasta, cache warm-up, shard

Warm the ESM checkpoint **single-threaded** and raise on failure -- two workers downloading it
concurrently corrupt the archive, and printing instead of raising re-creates the race.


In [ ]:
fadir = f'{SCRATCH}/fa'; os.makedirs(fadir, exist_ok=True)
for tid, t in TG.items():
    open(f'{fadir}/{tid}.fasta', 'w').write('>A|' + tid + '\n' + t['hseq'] + '\n')

w = subprocess.run([sys.executable, '-c',
    "import torch, esm; torch.hub.load('facebookresearch/esm:main','esm2_t36_3B_UR50D')"],
    capture_output=True, text=True)
if w.returncode:
    raise RuntimeError('esm warm-up failed; aborting rather than racing:\n' + (w.stdout + w.stderr)[-2000:])

tids = sorted(TG)
for g in (0, 1):
    d = f'{SCRATCH}/sh{g}'; shutil.rmtree(d, ignore_errors=True); os.makedirs(d)
for i, tid in enumerate(tids):
    os.link(f'{fadir}/{tid}.fasta', f'{SCRATCH}/sh{i % 2}/{tid}.fasta')
print('warmed;', len(tids), 'fastas sharded', flush=True)
note('warmed')


## Scoring helpers

Framework-aligned CDR-H3 RMSD is the primary read-out here: it isolates loop error instead of
letting a displaced loop drag the superposition toward itself. Chain-aligned is reported too so the
numbers stay comparable with the 597-target benchmark.

Validity is measured on every sample, because RMSD hides broken geometry.


In [ ]:
import biotite.structure.io.pdbx as pdbx
AT = ['N', 'CA', 'C', 'CB']

def kabsch(P, Q):
    Pm, Qm = P.mean(0), Q.mean(0)
    V, _, W = np.linalg.svd((P - Pm).T @ (Q - Qm))
    D = np.diag([1, 1, np.sign(np.linalg.det(V @ W))])
    return V @ D @ W, Pm, Qm

rmsd = lambda a, b: float(np.sqrt(((a - b) ** 2).sum(-1).mean()))

MASKS = {}
for tid, t in TG.items():
    Ln = len(t['hseq']); lo, hi = t['cdr3']
    c3m = np.zeros(Ln, bool); c3m[lo:hi] = True
    fw = np.ones(Ln, bool)
    for k in ('cdr1', 'cdr2', 'cdr3'):
        a_, b_ = t[k]; fw[a_:b_] = False
    ref = np.array([[[np.nan] * 3 if v is None else v for v in r] for r in t['hca']], float)
    MASKS[tid] = (Ln, c3m, fw, ref, int(hi - lo))

def parse(fp, Ln):
    """Return (Ln,4,3) array, or None if the file is not one chain numbered 1..Ln."""
    st = pdbx.get_structure(pdbx.CIFFile.read(fp), model=1)
    u = np.unique(st.res_id)
    if len(np.unique(st.chain_id)) != 1 or len(u) != Ln or u.max() - u.min() + 1 != Ln:
        return None
    P = np.full((Ln, 4, 3), np.nan)
    rid = st.res_id - st.res_id.min()
    for k, an in enumerate(AT):
        m = st.atom_name == an
        P[rid[m], k] = st.coord[m]
    return P

def score(P, tid):
    """(fw-aligned CDR-H3 rmsd, chain-aligned CDR-H3 rmsd) or (None, None)."""
    Ln, c3m, fw, ref, _ = MASKS[tid]
    ok = np.isfinite(ref[..., 0]) & np.isfinite(P[..., 0])
    s2 = ok & c3m[:, None]
    if s2.sum() < 3:
        return None, None
    out = []
    for sel in (ok & fw[:, None], ok):
        if sel.sum() < 60:
            out.append(None); continue
        Rot, Pm, Qm = kabsch(P[sel], ref[sel])
        al = np.full_like(P, np.nan); al[ok] = (P[ok] - Pm) @ Rot + Qm
        out.append(rmsd(al[s2], ref[s2]))
    return out[0], out[1]

def validity(P):
    """Peptide-bond, CA-CA and CA-clash violations. Broken chains must not count as hits."""
    N, CA, C = P[:, 0], P[:, 1], P[:, 2]
    pep = np.linalg.norm(C[:-1] - N[1:], axis=-1)
    caca = np.linalg.norm(CA[:-1] - CA[1:], axis=-1)
    d = np.linalg.norm(CA[:, None] - CA[None, :], axis=-1)
    sep = np.abs(np.arange(len(CA))[:, None] - np.arange(len(CA))[None, :])
    clash = int(np.nansum((d < 3.5) & (sep >= 3)) // 2)
    bad = lambda v, lo, hi: float(np.nanmean((v < lo) | (v > hi)))
    return {'bad_pep': round(bad(pep, 1.2, 1.5), 4),
            'bad_caca': round(bad(caca, 3.4, 4.1), 4),
            'clash': clash}
print('scoring helpers ready', flush=True)


## The tau sweep

One arm at a time: fold on both GPUs, score, record, delete the structures. A timeout therefore
leaves completed arms intact in `hitrate_results.json`.


In [ ]:
THRESH = [1.5, 2.0, 2.5, 3.0]
KS = [1, 2, 4, 8, 12, 16, 20]

def fold(tau, tag):
    t0 = time.time(); procs = []
    for g in (0, 1):
        if g:
            time.sleep(60)                       # stagger: both load fp32 ESM into host RAM first
        procs.append(subprocess.Popen(
            ['simplefold', '--simplefold_model', MODEL, '--backend', 'torch',
             '--ckpt_dir', f'{SCRATCH}/artifacts',
             '--num_steps', STEPS, '--tau', tau, '--nsample_per_protein', NS, '--seed', SEED,
             '--fasta_path', f'{SCRATCH}/sh{g}', '--output_dir', f'{SCRATCH}/{tag}_out{g}'],
            stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
            env={**os.environ, 'CUDA_VISIBLE_DEVICES': str(g)}))
        print(f'  launched gpu{g} tau={tau}', flush=True)
    rc = {}
    for g, p in enumerate(procs):
        o = p.communicate()[0]; rc[f'gpu{g}'] = p.returncode
        if p.returncode:
            print(f'  GPU{g} FAILED tau={tau}\n', o[-2000:], flush=True)
    return rc, round((time.time() - t0) / 60, 1)

for tau in TAUS:
    tag = 't' + tau.replace('.', '')
    print(f'=== tau {tau} ===', flush=True)
    rc, fmin = fold(tau, tag)
    rows, skip = [], []
    for tid in sorted(TG):
        Ln, c3m, fw, ref, l3 = MASKS[tid]
        files = sorted(glob.glob(f'{SCRATCH}/{tag}_out*/**/*{tid}*.cif', recursive=True))
        fwe, che, val = [], [], []
        for fp in files:
            P = parse(fp, Ln)
            if P is None:
                skip.append(f'{tid}:numbering'); continue
            f_, c_ = score(P, tid)
            if f_ is None:
                skip.append(f'{tid}:score'); continue
            fwe.append(f_); che.append(c_); val.append(validity(P))
        if not fwe:
            skip.append(f'{tid}:nofiles'); continue
        fwe = np.array(fwe); che = np.array(che)
        rows.append({
            'tid': tid, 'group': 'hard' if tid in HARD else 'easy', 'cdr3_len': l3,
            'n': len(fwe),
            'fw_best': round(float(fwe.min()), 3), 'fw_med': round(float(np.median(fwe)), 3),
            'ch_best': round(float(che.min()), 3), 'ch_med': round(float(np.median(che)), 3),
            'fw_all': [round(float(x), 3) for x in fwe],
            'bad_pep': round(float(np.mean([v['bad_pep'] for v in val])), 4),
            'bad_caca': round(float(np.mean([v['bad_caca'] for v in val])), 4),
            'clash': round(float(np.mean([v['clash'] for v in val])), 1)})

    arm = {'tau': float(tau), 'rc': rc, 'fold_min': fmin, 'n_scored': len(rows),
           'skip': skip, 'rows': rows}
    for grp in ('hard', 'easy'):
        sub = [r for r in rows if r['group'] == grp]
        if not sub:
            continue
        a = {'n': len(sub)}
        for th in THRESH:
            a[f'hit@{th}'] = round(sum(1 for r in sub if r['fw_best'] < th) / len(sub), 3)
        a['fw_best_med'] = round(float(np.median([r['fw_best'] for r in sub])), 3)
        a['fw_med_med'] = round(float(np.median([r['fw_med'] for r in sub])), 3)
        a['ch_med_med'] = round(float(np.median([r['ch_med'] for r in sub])), 3)
        # best-of-k curve: mean over targets of the median best-of-k across 200 random draws
        rng = np.random.default_rng(0)
        curve = {}
        for k in KS:
            vals = []
            for r in sub:
                arr = np.array(r['fw_all'])
                if len(arr) < k:
                    continue
                vals.append(float(np.median([arr[rng.choice(len(arr), k, replace=False)].min()
                                             for _ in range(200)])))
            if vals:
                curve[k] = round(float(np.median(vals)), 3)
        a['best_of_k'] = curve
        a['bad_pep'] = round(float(np.mean([r['bad_pep'] for r in sub])), 4)
        a['bad_caca'] = round(float(np.mean([r['bad_caca'] for r in sub])), 4)
        a['clash'] = round(float(np.mean([r['clash'] for r in sub])), 1)
        arm[grp] = a
        print(f"  {grp}: n={a['n']} hit@2.0={a['hit@2.0']} best_med={a['fw_best_med']} "
              f"med_med={a['fw_med_med']} clash={a['clash']} bad_pep={a['bad_pep']}", flush=True)

    R['arms'][tau] = arm
    note(f'tau{tau}')
    for g in (0, 1):
        shutil.rmtree(f'{SCRATCH}/{tag}_out{g}', ignore_errors=True)
    print(f'  arm done in {fmin} min; structures deleted', flush=True)

note('done')
print('\nALL ARMS COMPLETE', R['elapsed_min'], 'min', flush=True)


## Verdict against the pre-registered kill criteria

In [ ]:
print(f"{'tau':>6} | {'hit@2 hard':>10} {'hit@2 easy':>10} | {'best med':>9} {'med med':>8} "
      f"| {'clash':>6} {'bad_pep':>8}")
for tau in TAUS:
    a = R['arms'].get(tau)
    if not a or 'hard' not in a:
        continue
    h, e = a['hard'], a.get('easy', {})
    print(f"{tau:>6} | {h['hit@2.0']:>10} {e.get('hit@2.0', '-'):>10} | "
          f"{h['fw_best_med']:>9} {h['fw_med_med']:>8} | {h['clash']:>6} {h['bad_pep']:>8}")

best = max((a['hard']['hit@2.0'], t) for t, a in R['arms'].items() if 'hard' in a)
hr, bt = best
R['verdict'] = {'best_tau': bt, 'hit@2.0': hr,
                'reading': ('MODE ABSENT - close all steering work' if hr < 0.10 else
                            'MODE RARE - guidance is live, use OT not L1' if hr <= 0.40 else
                            'SELECTION PROBLEM - re-test selection first')}
print('\nbest tau', bt, ' hit@2.0 =', hr, '\n->', R['verdict']['reading'])
ez = R['arms'][bt].get('easy', {}).get('hit@2.0')
print('control (easy) hit@2.0 at same tau =', ez,
      '  <- if this is also low the metric or sampler is broken, not the prior')
note('done')
